In [1]:
from IPython.display import display, HTML
from pyecharts.globals import CurrentConfig, NotebookType

CurrentConfig.NOTEBOOK_TYPE = NotebookType.JUPYTER_LAB
CurrentConfig.ONLINE_HOST = "https://cdn.bootcdn.net/ajax/libs/echarts/5.5.0/"
#CurrentConfig.ONLINE_HOST = "https://cdn.jsdelivr.net/npm/echarts@5/dist/"
#CurrentConfig.ONLINE_HOST = 'https://cdn.jsdelivr.net/gh/pyecharts/pyecharts-assets@master/assets/'# 或者直接修改库的default host

In [2]:
import html
from IPython.display import HTML

def render_in_iframe(chart):
    """在 iframe 中渲染 pyecharts，绕过跟踪防护并自适应图表尺寸"""
    # 获取图表的 HTML 片段
    html_content = chart.render_embed()
    
    # 优化 1：使用标准库进行更安全、彻底的 HTML 转义
    # 相比简单的 replace('"', '&quot;')，这能防止其他特殊字符破坏 iframe 结构
    escaped_html = html.escape(html_content)
    
    # 优化 2：动态获取图表的宽高
    # 避免写死 500px。如果你的图表在初始化时设置了 init_opts=opts.InitOpts(height="800px")，这里能自动适配
    width = getattr(chart, 'width', '100%')
    height = getattr(chart, 'height', '500px')
    
    # 优化 3：增加 scrolling 和 sandbox 属性提升体验和安全性
    iframe_html = f"""
    <iframe srcdoc="{escaped_html}" 
            width="{width}" 
            height="{height}" 
            frameborder="0"
            scrolling="no"
            sandbox="allow-scripts">
    </iframe>
    """
    return HTML(iframe_html)
# 使用
#render_in_iframe(bar)

In [3]:
from pyecharts.charts import Bar
from pyecharts import options as opts

bar = Bar()
bar.add_xaxis(["衬衫", "毛衣", "领带", "裤子"])
bar.add_yaxis("商家A", [114, 55, 27, 101])
bar.set_global_opts(title_opts=opts.TitleOpts(title="测试"))

#render_in_iframe(bar)
bar.render_notebook()

In [ ]:
#render_in_iframe(bar)

In [ ]:
!jupyter nbconvert --to html test.ipynb

In [ ]:
def render_chart(chart):
    _ensure_echarts_loaded()
    
    # 获取图表配置
    options = json.loads(chart.dump_options())
    width = getattr(chart, '_width', '100%')
    height = getattr(chart, '_height', '500px')
    
    # 生成 iframe
    #沙盒sandbox="allow-scripts allow-same-origin"存在安全问题
    iframe = f"""
    <iframe 
        width="{width}" 
        height="{height}" 
        frameborder="0"
        scrolling="no"
        style="border: none;"
        sandbox="allow-scripts allow-same-origin"
        srcdoc='
        <!DOCTYPE html>
        <html>
        <head><meta charset="utf-8"><style>body{{margin:0;padding:0;}}</style></head>
        <body>
            <div id="chart" style="width:{width};height:{height};"></div>
            <script>
                function init() {{
                    var echarts = window.parent.echarts;
                    if (!echarts) {{
                        if (window.parent.__echarts_loading) {{
                            setTimeout(init, 100);
                            return;
                        }}
                        var script = document.createElement("script");
                        script.src = "{CurrentConfig.ONLINE_HOST}echarts.min.js";
                        script.onload = function() {{
                            echarts = window.echarts;
                            var chart = echarts.init(document.getElementById("chart"));
                            chart.setOption({json.dumps(options)});
                        }};
                        document.head.appendChild(script);
                    }} else {{
                        var chart = echarts.init(document.getElementById("chart"));
                        chart.setOption({json.dumps(options)});
                        window.addEventListener("resize", () => chart.resize());
                    }}
                }}
                init();
            </script>
        </body>
        </html>
        '
    ></iframe>
    """
    return HTML(iframe)